# Two-stage stochastic linear programs and the L-shaped method

This notebook builds a **generic** implementation of the L-shaped method: the algorithm is written
once, in terms of the data $(c, A, b, q, W, T(\xi), h(\xi))$ of the slides, and an instance is
nothing but a value of that data. We use it on an ice-cream capacity investment problem, and then
re-use the very same code, unchanged, on a second instance.

## The problem

Consider the optimal capacity investment in various types of ice-cream production plants.
Four plants are considered and they can produce three different ice-cream flavors.
The demand in the next period has to be satisfied for each of the three flavors, and is equal to
some random value $\xi$ for flavor 1, 3 for flavor 2, and 2 for flavor 3.
$\xi$ is discrete with three realizations 3, 5, 7, associated to the probabilities 0.3, 0.4 and 0.3
respectively.
There is a budget constraint on the investment and also a constraint on the minimum total capacity:
the minimum total capacity to be installed is 12 and we have a budget limit of 120.
The unit capacity costs for the four plants are 10, 7, 16 and 6, respectively.
The cost per production unit of plant $i$ for flavor $j$ is denoted $a_{ij}$, and gathering all the
costs, we can construct the matrix

$$
A = \begin{pmatrix}
  40 & 24 & 4 \\
  45 & 27 & 4.5 \\
  32 & 19.2 & 3.2 \\
  55 & 33 & 5.5
\end{pmatrix}
$$

We aim to minimize the total cost (investment and production costs).
The capacity $x_i$ installed in plant $i$ is a **first-stage** decision, taken before the demand is
known; the production plan $y_{ij}$ is a **second-stage** decision, taken once $\xi$ is observed:

\begin{align*}
    \min_x\ & \sum_{i=1}^4 c_i x_i + \mathbb{E}_{\xi}[Q(x,\xi)] \\
    \mbox{s.t. } & \sum_{i=1}^{4} x_i \geq 12 \\
                 & \sum_{i=1}^4 c_i x_i \leq 120 \\
                 & x \geq 0
\end{align*}

where

\begin{align*}
Q(x,\xi) = \min_y\ & \sum_{i,j} a_{ij} y_{ij} \\
\mbox{s.t. } & \sum_{j=1}^3 y_{ij} \leq x_i,\ \forall\, i \\
             & \sum_{i=1}^4 y_{ij} \geq d_j(\xi),\ \forall\, j \\
             & y \geq 0
\end{align*}

In [ ]:
using JuMP
using LinearAlgebra
using Printf
using HiGHS

const SOLVER = HiGHS.Optimizer

## A generic two-stage linear program

Rather than hard-coding the ice-cream model, we store the data of the standard form used in the
course:

$$
\min_{x \geq 0} \left\{ c^Tx + \mathbb{E}_{\boldsymbol{\xi}}[Q(x,\boldsymbol{\xi})] \ \middle|\ Ax \geq b \right\},
\qquad
Q(x,\xi) = \min_{y \geq 0} \left\{ q^Ty \ \middle|\ Wy \geq h(\xi) - T(\xi)x \right\}.
$$

Two conventions make everything that follows short and uniform.

* **Every constraint is written as $\geq$.** An equality is split into two inequalities and a
  $\leq$ row is multiplied by $-1$. The price is a few extra rows; the reward is that all the dual
  multipliers of the second stage have the *same* sign. JuMP returns non-negative multipliers for
  $\geq$ rows of a minimization problem, so the cut formulas below can be transcribed literally from
  the slides, with no sign bookkeeping.
* **$T$ and $h$ are functions of the scenario.** Passing a matrix instead of a function is allowed
  and means *fixed recourse* in that component.

The matrices are only required to be `AbstractMatrix{Float64}`, so a `SparseMatrixCSC` from
`SparseArrays` can be passed instead of a dense matrix on larger instances; we keep dense matrices
here, the instances being tiny.

In [ ]:
"""
    TwoStageLP(; c, A, b, q, W, T, h, ξ, p)

Data of a two-stage stochastic linear program

    min  cᵀx + E[Q(x,ξ)]   s.t.  Ax ≥ b,  x ≥ 0
    Q(x,ξ) = min qᵀy       s.t.  Wy ≥ h(ξ) - T(ξ)x,  y ≥ 0

`T` and `h` may be given either as functions of the scenario or as a constant matrix/vector.
`ξ` is the vector of scenarios (numbers or vectors) and `p` the associated probabilities.
"""
struct TwoStageLP
    c::Vector{Float64}          # first-stage cost
    A::AbstractMatrix{Float64}  # first-stage constraints, Ax ≥ b
    b::Vector{Float64}
    q::Vector{Float64}          # second-stage cost
    W::AbstractMatrix{Float64}  # recourse matrix
    T::Function                 # ξ ↦ T(ξ), technology matrix
    h::Function                 # ξ ↦ h(ξ), right-hand side
    ξ::Vector                   # scenarios
    p::Vector{Float64}          # probabilities
end

function TwoStageLP(; c, A, b, q, W, T, h, ξ, p)
    Tf = T isa AbstractMatrix ? (_ -> Float64.(T)) : T
    hf = h isa AbstractVector ? (_ -> Float64.(h)) : h
    pb = TwoStageLP(Float64.(c), Float64.(A), Float64.(b), Float64.(q), Float64.(W),
                    Tf, hf, collect(ξ), Float64.(p))
    # a few sanity checks: a wrong dimension here is much harder to diagnose later on
    @assert size(pb.A, 2) == length(pb.c) "A and c disagree on the number of first-stage variables"
    @assert size(pb.A, 1) == length(pb.b) "A and b disagree on the number of first-stage rows"
    @assert size(pb.W, 2) == length(pb.q) "W and q disagree on the number of second-stage variables"
    @assert length(pb.ξ) == length(pb.p)  "one probability per scenario is required"
    @assert isapprox(sum(pb.p), 1; atol = 1e-9) "the probabilities must sum up to one"
    for s in pb.ξ
        @assert size(pb.T(s), 1) == size(pb.W, 1) "T(ξ) and W disagree on the number of rows"
        @assert size(pb.T(s), 2) == length(pb.c)  "T(ξ) and c disagree on the number of columns"
        @assert length(pb.h(s)) == size(pb.W, 1)  "h(ξ) and W disagree on the number of rows"
    end
    return pb
end

n_x(pb::TwoStageLP)         = length(pb.c)       # first-stage variables
n_y(pb::TwoStageLP)         = length(pb.q)       # second-stage variables
n_rows(pb::TwoStageLP)      = size(pb.W, 1)      # second-stage constraints
n_scenarios(pb::TwoStageLP) = length(pb.ξ)

### Writing the ice-cream instance in that form

The second stage has $4 \times 3$ variables, which we store flavor by flavor:
$y = (y_{11}, y_{12}, y_{13}, y_{21}, \ldots, y_{43})$, so that $y_{ij}$ sits in position
$3(i-1)+j$.

The seven second-stage rows are

* four **capacity** rows, $\sum_j y_{ij} \leq x_i$, rewritten $-\sum_j y_{ij} \geq -x_i$: the
  corresponding row of $W$ carries $-1$ on the block of plant $i$, $h_i = 0$ and $T_{ii} = 1$;
* three **demand** rows, $\sum_i y_{ij} \geq d_j(\xi)$: the row of $W$ carries $+1$ on flavor $j$,
  $T$ is null and $h$ holds the demand, the first component being the random one.

The first stage has the minimum capacity row and the budget row, the latter being reversed:
$\sum_i c_i x_i \leq 120 \iff -\sum_i c_i x_i \geq -120$.

In [ ]:
const NPLANTS = 4
const NFLAVORS = 3

opening_costs = [10.0, 7.0, 16.0, 6.0]
production_costs = [40.0 24.0 4.0; 45.0 27.0 4.5; 32.0 19.2 3.2; 55.0 33.0 5.5]

"""Data of the ice-cream problem, written in the standard form above."""
function icecream_data(; min_capacity = 12.0, budget = 120.0, other_demands = [3.0, 2.0])
    ny = NPLANTS * NFLAVORS
    idx(i, j) = NFLAVORS * (i - 1) + j          # position of y_ij in the flat vector

    W = zeros(NPLANTS + NFLAVORS, ny)
    T = zeros(NPLANTS + NFLAVORS, NPLANTS)
    for i in 1:NPLANTS, j in 1:NFLAVORS
        W[i, idx(i, j)] = -1.0                  # capacity:  -Σ_j y_ij ≥ -x_i
        W[NPLANTS + j, idx(i, j)] = 1.0         # demand:     Σ_i y_ij ≥ d_j(ξ)
    end
    for i in 1:NPLANTS
        T[i, i] = 1.0
    end
    h(ξ) = vcat(zeros(NPLANTS), ξ, other_demands)

    return TwoStageLP(
        c = opening_costs,
        A = [ones(1, NPLANTS); -opening_costs'],       # Σ x_i ≥ min_capacity, -cᵀx ≥ -budget
        b = [min_capacity, -budget],
        q = vec(permutedims(production_costs)),        # flattened row by row, as idx expects
        W = W, T = T, h = h,
        ξ = [3.0, 5.0, 7.0], p = [0.3, 0.4, 0.3])
end

icecream = icecream_data()

## The extensive form

We start with the extensive form, which will serve as a reference solution. Note that it is written
once and for all: it only manipulates $(c, A, b, q, W, T, h)$.

In [ ]:
"""Build and solve the extensive form (deterministic equivalent) of `pb`."""
function extensive_form(pb::TwoStageLP; optimizer = SOLVER, silent = true)
    S = n_scenarios(pb)
    m = Model(optimizer)
    silent && set_silent(m)

    @variable(m, x[1:n_x(pb)] >= 0)
    @variable(m, y[1:n_y(pb), 1:S] >= 0)          # one recourse vector per scenario

    @constraint(m, pb.A * x .>= pb.b)
    for s in 1:S
        @constraint(m, pb.T(pb.ξ[s]) * x + pb.W * y[:, s] .>= pb.h(pb.ξ[s]))
    end

    @objective(m, Min, dot(pb.c, x) + sum(pb.p[s] * dot(pb.q, y[:, s]) for s in 1:S))

    optimize!(m)
    @assert termination_status(m) == MOI.OPTIMAL "extensive form: $(termination_status(m))"
    return m, value.(x), objective_value(m)
end

In [ ]:
ef, x_ef, obj_ef = extensive_form(icecream)
@printf("extensive form: optimal value = %.6f\n", obj_ef)
println("x* = ", x_ef)

The number of variables of the extensive form grows linearly with the number of scenarios, which is
what the L-shaped method avoids.

## The second-stage programs

The second stage is solved once per scenario and per iteration, but only its **right-hand side**
$h(\xi) - T(\xi)x$ changes. We therefore build the model **once** and update the right-hand side
afterwards, which is both faster and closer to what a real implementation does.

In [ ]:
"""A JuMP model whose right-hand side is meant to be updated in place."""
struct StageProblem{C}
    model::Model
    y::Vector{VariableRef}
    con::Vector{C}          # the rows Wy ≥ ⋅ , whose duals give the cut coefficients
end

"""Recourse problem  min qᵀy s.t. Wy ≥ ⋅, y ≥ 0, with a placeholder right-hand side."""
function second_stage(pb::TwoStageLP; optimizer = SOLVER)
    m = Model(optimizer)
    set_silent(m)
    @variable(m, y[1:n_y(pb)] >= 0)
    con = @constraint(m, pb.W * y .>= zeros(n_rows(pb)))
    @objective(m, Min, dot(pb.q, y))
    return StageProblem(m, y, con)
end

"""Set the right-hand side to h(ξ) - T(ξ)x and re-optimize. Returns the termination status."""
function solve_recourse!(sp::StageProblem, pb::TwoStageLP, x, ξ)
    set_normalized_rhs.(sp.con, pb.h(ξ) - pb.T(ξ) * x)
    optimize!(sp.model)
    return termination_status(sp.model)
end

### A word of caution on the sign of the multipliers

The comment `# ≥ 0: every row is a ≥ row of a Min problem` attached to `dual.(recourse.con)` in the
algorithm below is a consequence of **our modelling choice**, not a general fact — and getting it
wrong is probably the most common bug in a hand-written L-shaped code.

JuMP follows the conic duality convention of MathOptInterface, in which the sign returned by `dual`
depends on *two* things: the sense of the optimization, and the set the constraint lives in.

| sense | `≥` (`GreaterThan`) | `≤` (`LessThan`) | `==` (`EqualTo`) |
|:--|:--:|:--:|:--:|
| `Min` | $\pi \geq 0$ | $\pi \leq 0$ | free |
| `Max` | $\pi \leq 0$ | $\pi \geq 0$ | free |

A few consequences worth keeping in mind.

* Writing the capacity constraint in its natural form `sum(y[i, :]) <= x[i]` is perfectly correct,
  but its multiplier is then **non-positive**, and the corresponding row of $T$ has to carry the
  matching sign. If a model mixes $\leq$ and $\geq$ rows inside the same $(W, T, h)$ — as the
  earlier version of this notebook did — then `dual.(con)` returns a vector with *mixed* signs, and
  every formula using it must be read component by component. Writing all the rows as $\geq$ is
  precisely what makes the single expression `E .+= p[s] .* (T(ξ)' * π)` correct for all rows at
  once.
* The duality of the course, $Q(x,\xi) = \max_{\pi} \lbrace \pi^T(h(\xi) - T(\xi)x) : W^T\pi \leq q \rbrace$,
  is stated for one particular primal form. A JuMP model departing from that form still returns
  correct multipliers; they are simply expressed in another convention, and comparing them to the
  slides without care is a good way to convince yourself that the code is wrong when it is not.
* `dual` is not the only convention on offer: JuMP also provides `shadow_price`, which reports the
  marginal change of the objective value under a relaxation of the constraint, and which therefore
  does not always agree in sign with `dual`. Pick one and stay with it; everything here uses `dual`.

The moral is not to memorize the table, but to **check numerically**. Two identities cost one line
each and catch a sign error immediately: strong duality,

$$Q(x,\xi) = \pi^T(h(\xi) - T(\xi)x),$$

and the fact that the cut is a supporting hyperplane of $Q(\cdot,\xi)$ at $x$,

$$Q(x',\xi) \geq \pi^Th(\xi) - (T(\xi)^T\pi)^Tx' \quad \forall x', \qquad
\text{with equality at } x' = x.$$

In [ ]:
# Checks on a single scenario: signs, strong duality, and the supporting hyperplane.
# Note that a degenerate second stage may admit several optimal dual solutions, so the
# multipliers themselves are solver-dependent: it is the three properties that must hold.
rec = second_stage(icecream)
x_test = fill(3.0, NPLANTS)
ξ_test = 5.0
@assert solve_recourse!(rec, icecream, x_test, ξ_test) == MOI.OPTIMAL

π_test = dual.(rec.con)
r_test = icecream.h(ξ_test) - icecream.T(ξ_test) * x_test
Q_test = objective_value(rec.model)

println("multipliers          : ", π_test)
println("all non-negative     : ", all(π_test .>= -1e-9))
@printf("Q(x,ξ)               : %.6f\n", Q_test)
@printf("πᵀ(h(ξ) - T(ξ)x)     : %.6f\n", dot(π_test, r_test))

# the cut built at x_test must under-estimate Q(·,ξ) everywhere, and be tight at x_test
cut(x) = dot(π_test, icecream.h(ξ_test)) - dot(icecream.T(ξ_test)' * π_test, x)
@printf("cut(x_test) - Q(x_test,ξ) = %.3e\n", cut(x_test) - Q_test)
for x in ([3.0, 9.0, 8.0, 2.0], [9.0, 1.0, 9.0, 0.0], [8.0, 7.0, 6.0, 10.0])
    @assert solve_recourse!(rec, icecream, x, ξ_test) == MOI.OPTIMAL
    Qx = objective_value(rec.model)
    @printf("  x = %-22s Q = %9.3f   cut = %9.3f   cut ≤ Q: %s\n",
            string(x), Qx, cut(x), cut(x) <= Qx + 1e-9)
end

### The feasibility subproblem

When the recourse problem is infeasible for some scenario, we need a vector $\sigma$ proving it.
We obtain it from the feasibility subproblem

$$
\min_{y, w \geq 0} \left\{ \mathbf{1}^Tw \ \middle|\ Wy + w \geq h(\xi) - T(\xi)x \right\},
$$

whose optimal value is $0$ if and only if the recourse problem is feasible; its dual multipliers
$\sigma$ then provide the feasibility cut. With all the rows written as $\geq$, a single artificial
variable per row suffices.

Some solvers directly return an infeasibility certificate for the recourse problem, which is exactly
such a $\sigma$ and saves one LP; we use it when it is available and fall back on the subproblem
otherwise.

In [ ]:
"""Feasibility subproblem  min 1ᵀw s.t. Wy + w ≥ ⋅, y, w ≥ 0."""
function feasibility_stage(pb::TwoStageLP; optimizer = SOLVER)
    m = Model(optimizer)
    set_silent(m)
    @variable(m, y[1:n_y(pb)] >= 0)
    @variable(m, w[1:n_rows(pb)] >= 0)
    con = @constraint(m, pb.W * y + w .>= zeros(n_rows(pb)))
    @objective(m, Min, sum(w))
    return StageProblem(m, y, con)
end

"""Multipliers σ ≥ 0 certifying that the recourse problem is infeasible at (x, ξ)."""
function infeasibility_multipliers!(fs::StageProblem, rec::StageProblem, pb::TwoStageLP, x, ξ)
    if dual_status(rec.model) == MOI.INFEASIBILITY_CERTIFICATE
        # the solver already gives a dual ray; for ≥ rows of a Min problem it is non-negative
        return dual.(rec.con)
    end
    set_normalized_rhs.(fs.con, pb.h(ξ) - pb.T(ξ) * x)
    optimize!(fs.model)
    return dual.(fs.con)
end

## The L-shaped method

At iteration $k$ the master problem is

$$
\min_{x \geq 0,\ \theta} \left\{ c^Tx + \theta \ \middle|\ Ax \geq b, \ \text{cuts} \right\},
$$

$\theta$ being left out of the objective as long as no optimality cut has been generated.
Solving the second stage at $x^k$ gives, per scenario, either

* multipliers $\pi_s \geq 0$, and we accumulate

  $$E = \sum_s p_s T(\xi_s)^T \pi_s, \qquad e = \sum_s p_s h(\xi_s)^T \pi_s,$$

  which yields the **optimality cut** $E^Tx + \theta \geq e$, a supporting hyperplane of
  $\mathcal{Q}$ at $x^k$;
* or multipliers $\sigma \geq 0$ of the feasibility subproblem, which yield the **feasibility cut**
  $(T(\xi_s)^T\sigma)^Tx \geq \sigma^Th(\xi_s)$, i.e. $\sigma^T(h(\xi_s) - T(\xi_s)x) \leq 0$.

We stop when $\theta^k \geq \mathcal{Q}(x^k)$, up to a tolerance.

In [ ]:
"""
    lshaped(pb; maxiter, tol, verbose)

Single-cut L-shaped method. Returns a named tuple with the optimal solution, the optimal value,
the number of cuts of each kind, the master problem and the iteration history.
"""
function lshaped(pb::TwoStageLP; optimizer = SOLVER, maxiter = 100, tol = 1e-8, verbose = true)
    n = n_x(pb)

    master = Model(optimizer)
    set_silent(master)
    @variable(master, x[1:n] >= 0)
    @variable(master, θ)
    @constraint(master, pb.A * x .>= pb.b)
    @objective(master, Min, dot(pb.c, x))   # θ enters only with the first optimality cut

    recourse = second_stage(pb; optimizer)  # both models are built once
    feasible_pb = feasibility_stage(pb; optimizer)

    n_opt = n_feas = 0
    history = NamedTuple[]

    verbose && println(" iter   lower bound   upper bound             θ          Q(x)")

    for k in 1:maxiter
        # ---- 1. master problem ------------------------------------------------------------
        optimize!(master)
        termination_status(master) == MOI.OPTIMAL ||
            error("master problem: $(termination_status(master))")
        xk = value.(x)
        θk = n_opt > 0 ? value(θ) : -Inf
        # the master value is a valid lower bound only once θ is part of the objective
        lb = n_opt > 0 ? objective_value(master) : -Inf

        # ---- 2. second-stage programs -----------------------------------------------------
        Q, E, e, cut_added = 0.0, zeros(n), 0.0, false
        for s in 1:n_scenarios(pb)
            ξs = pb.ξ[s]
            status = solve_recourse!(recourse, pb, xk, ξs)

            if status == MOI.INFEASIBLE
                σ = infeasibility_multipliers!(feasible_pb, recourse, pb, xk, ξs)
                @constraint(master, dot(pb.T(ξs)' * σ, x) >= dot(σ, pb.h(ξs)))
                n_feas += 1
                cut_added = true
                verbose && @printf("%5d   feasibility cut (scenario %d)\n", k, s)
                break                       # a new x is needed before going on
            elseif status != MOI.OPTIMAL
                error("second stage, scenario $s: $status")
            end

            π = dual.(recourse.con)         # ≥ 0: every row is a ≥ row of a Min problem
            Q += pb.p[s] * objective_value(recourse.model)
            E .+= pb.p[s] .* (pb.T(ξs)' * π)
            e += pb.p[s] * dot(π, pb.h(ξs))
        end
        cut_added && continue

        # ---- 3. optimality cut, or stop ---------------------------------------------------
        ub = dot(pb.c, xk) + Q              # cᵀxᵏ + Q(xᵏ) is attainable, hence an upper bound
        push!(history, (iteration = k, lb = lb, ub = ub, θ = θk, Q = Q, x = xk))
        if verbose
            # θ and the master value only become meaningful once θ is in the objective
            lb_str = n_opt > 0 ? @sprintf("%.6f", lb) : "-Inf"
            θ_str = n_opt > 0 ? @sprintf("%.6f", θk) : "-Inf"
            @printf("%5d   %11s   %11.6f   %11s   %11.6f\n", k, lb_str, ub, θ_str, Q)
        end

        if θk >= Q - tol
            verbose && @printf("converged in %d iterations (%d optimality cuts, %d feasibility cuts)\n",
                               k, n_opt, n_feas)
            return (x = xk, objective = ub, iterations = k,
                    optimality_cuts = n_opt, feasibility_cuts = n_feas,
                    master = master, history = history)
        end

        @constraint(master, dot(E, x) + θ >= e)
        if n_opt == 0
            @objective(master, Min, dot(pb.c, x) + θ)   # θ is now bounded from below
        end
        n_opt += 1
    end
    error("no convergence in $maxiter iterations")
end

In [ ]:
result = lshaped(icecream)
println()
println("x*  = ", result.x)
@printf("obj = %.6f   (extensive form: %.6f, gap %.2e)\n",
        result.objective, obj_ef, abs(result.objective - obj_ef))

The two approaches agree. The exact optimum of this instance is
$x^\star = (8/3,\ 4,\ 10/3,\ 2)$, for a total expected cost of $28639/75 \approx 381.8533$: both the
minimum capacity and the budget constraints are active.

In [ ]:
println(result.master)

### Forcing feasibility cuts

With the minimum capacity constraint in place, the first master solution already installs 12 units
in the cheapest plant, for which a production plan exists. Relaxing that constraint lets the master
answer $x = 0$, for which no production plan can meet the demand: the first iterations then produce
feasibility cuts, one per infeasible scenario.

In [ ]:
relaxed = icecream_data(min_capacity = 0.0)
result_relaxed = lshaped(relaxed)
println()
println("x*  = ", result_relaxed.x)
@printf("obj = %.6f, %d feasibility cuts\n", result_relaxed.objective, result_relaxed.feasibility_cuts)

The optimal solution is unchanged: the minimum capacity constraint was not active at the optimum,
it only affected the path followed by the algorithm.

## The value of information

The same data structure gives the quantities of the course: the recourse problem value $RP$, the
wait-and-see value $WS$, the expected value problem $EV$ with its solution $\overline{x}(\overline{\xi})$,
and the expected result of using that solution, $EEV$. From them,

$$EVPI = RP - WS \geq 0, \qquad VSS = EEV - RP \geq 0.$$

In [ ]:
"""Solve the deterministic problem obtained by fixing the scenario to `ξ`."""
function deterministic_problem(pb::TwoStageLP, ξ; optimizer = SOLVER)
    m = Model(optimizer)
    set_silent(m)
    @variable(m, x[1:n_x(pb)] >= 0)
    @variable(m, y[1:n_y(pb)] >= 0)
    @constraint(m, pb.A * x .>= pb.b)
    @constraint(m, pb.T(ξ) * x + pb.W * y .>= pb.h(ξ))
    @objective(m, Min, dot(pb.c, x) + dot(pb.q, y))
    optimize!(m)
    @assert termination_status(m) == MOI.OPTIMAL "deterministic problem: $(termination_status(m))"
    return value.(x), objective_value(m)
end

"""Expected recourse cost Q(x) = E[Q(x,ξ)] (returns Inf if x is infeasible for some scenario)."""
function expected_recourse(pb::TwoStageLP, x; optimizer = SOLVER)
    sp = second_stage(pb; optimizer)
    Q = 0.0
    for s in 1:n_scenarios(pb)
        solve_recourse!(sp, pb, x, pb.ξ[s]) == MOI.OPTIMAL || return Inf
        Q += pb.p[s] * objective_value(sp.model)
    end
    return Q
end

"""RP, WS, EV, EEV and the derived EVPI and VSS."""
function value_of_information(pb::TwoStageLP; optimizer = SOLVER)
    _, _, RP = extensive_form(pb; optimizer)
    WS = sum(pb.p[s] * deterministic_problem(pb, pb.ξ[s]; optimizer)[2]
             for s in 1:n_scenarios(pb))
    ξbar = sum(pb.p[s] * pb.ξ[s] for s in 1:n_scenarios(pb))
    xbar, EV = deterministic_problem(pb, ξbar; optimizer)
    EEV = dot(pb.c, xbar) + expected_recourse(pb, xbar; optimizer)
    return (RP = RP, WS = WS, EV = EV, EEV = EEV,
            EVPI = RP - WS, VSS = EEV - RP, x_EV = xbar)
end

In [ ]:
voi = value_of_information(icecream)
for k in (:EV, :WS, :RP, :EEV, :EVPI, :VSS)
    @printf("%-5s = %10.4f\n", k, getfield(voi, k))
end
println("mean-value solution: ", voi.x_EV)
@printf("chain EV ≤ WS ≤ RP ≤ EEV: %s\n", voi.EV <= voi.WS <= voi.RP <= voi.EEV)

Perfect information is worth about 1.69 here, and using the distribution rather than the mean
demand about 2.13: both are small compared to the total cost of about 381.85, which says that this
instance is not very sensitive to the uncertainty. Note that the mean-value solution
$\overline{x}(\overline{\xi})$ differs markedly from $x^\star$, even though its expected cost is
only slightly worse.

## Re-using the code on another instance

Nothing above mentions ice cream, so any other two-stage linear program can be fed to the same
functions. Take the example of the course,

$$Q(x,\xi) = \min_{y^+, y^- \geq 0} \left\{ y^+ + y^- \ \middle|\ y^+ - y^- = \xi - x \right\} = |x - \xi|,$$

with $\boldsymbol{\xi}$ uniform over $\{1, 2, 4\}$ and no first-stage cost. The equality is split
into $y^+ - y^- \geq \xi - x$ and $-(y^+ - y^-) \geq -(\xi - x)$, hence

$$W = \begin{pmatrix} 1 & -1 \\ -1 & 1\end{pmatrix}, \qquad
  T = \begin{pmatrix} 1 \\ -1 \end{pmatrix}, \qquad
  h(\xi) = \begin{pmatrix} \xi \\ -\xi \end{pmatrix}.$$

The upper bound $x \leq 10$ merely keeps the master problem bounded before the first cut.

In [ ]:
absolute_deviation = TwoStageLP(
    c = [0.0],
    A = reshape([-1.0], 1, 1), b = [-10.0],        # -x ≥ -10, i.e. x ≤ 10
    q = [1.0, 1.0],                                # y⁺, y⁻
    W = [1.0 -1.0; -1.0 1.0],
    T = reshape([1.0, -1.0], 2, 1),
    h = ξ -> [ξ, -ξ],
    ξ = [1.0, 2.0, 4.0], p = fill(1 / 3, 3))

res = lshaped(absolute_deviation)
println()
@printf("x* = %.6f, RP = %.6f\n", res.x[1], res.objective)

The recourse function is $\mathcal{Q}(x) = \frac{1}{3}(|x-1| + |x-2| + |x-4|)$, minimized at the
**median** of the scenarios, $x^\star = 2$, with $\mathcal{Q}(2) = 1$: this is the piecewise linear
function drawn in the slides.

In [ ]:
voi2 = value_of_information(absolute_deviation)
for k in (:EV, :WS, :RP, :EEV, :EVPI, :VSS)
    @printf("%-5s = %8.4f\n", k, getfield(voi2, k))
end
println("mean-value solution: ", voi2.x_EV, "   (ξ̄ = 7/3)")

We recover the values of the slides: with perfect information every scenario can be matched exactly,
so $WS = 0$ and $EVPI = RP = 1$, whereas the mean-value decision $\overline{x}(\overline{\xi}) = 7/3$
costs $EEV = \mathcal{Q}(7/3) = 10/9$, hence $VSS = 1/9 \approx 0.1111$. Here perfect information is
worth far more than a better use of the distribution.

## The dual of the second stage

Feasibility cuts are easier to read on the dual of the recourse problem,

$$
\max_{\pi \geq 0} \left\{ \pi^T(h(\xi) - T(\xi)x) \ \middle|\ W^T\pi \leq q \right\},
$$

whose feasible set does not depend on $x$: the primal is infeasible exactly when this dual is
unbounded, and the direction along which it grows is the $\sigma$ of the feasibility cut.

In [ ]:
"""Dual of the recourse problem; `bound` caps the multipliers to expose an unbounded ray."""
function second_stage_dual(pb::TwoStageLP, x, ξ; bound = Inf, optimizer = SOLVER)
    m = Model(optimizer)
    set_silent(m)
    @variable(m, π[1:n_rows(pb)] >= 0)
    isfinite(bound) && set_upper_bound.(π, bound)
    @constraint(m, pb.W' * π .<= pb.q)
    @objective(m, Max, dot(pb.h(ξ) - pb.T(ξ) * x, π))
    optimize!(m)
    return m, π
end

In [ ]:
# no capacity installed: the demand cannot be met, so the dual is unbounded
m_dual, dual_vars = second_stage_dual(icecream, zeros(NPLANTS), 3.0)
println("status without bounds: ", termination_status(m_dual))

In [ ]:
# bounding the multipliers turns the ray into an ordinary optimal solution
m_dual, dual_vars = second_stage_dual(icecream, zeros(NPLANTS), 3.0; bound = 1e6)
σ = value.(dual_vars)
println("σ / ‖σ‖∞ = ", σ / norm(σ, Inf))

Normalizing gives a direction of recession of the dual feasible set, i.e. exactly the multipliers the
feasibility subproblem returns, and the cut it induces is
$\sigma^T(h(\xi) - T(\xi)x) \leq 0$.